# Chapter 5 explore: Your First LoRA Fine-Tune

Interactive companion to `code/chapter_05/first_lora_finetune.py`. Fine-tunes a LoRA adapter on Chapter 2's 16 training examples, then compares training recall against generalization on Chapter 2's held-out report. The full run takes about 5 minutes on CPU.

In [ ]:
import sys
sys.path.insert(0, "../code/chapter_01")
sys.path.insert(0, "../code/chapter_02")
sys.path.insert(0, "../code/chapter_03")
sys.path.insert(0, "../code/chapter_05")

from load_local_model import MODEL_NAME, load_model_and_tokenizer
from build_training_examples import build_training_examples, build_examples_for_report, SAMPLE_SET_DIR, HELD_OUT_REPORT
from baseline_prompting import run_baseline
from first_lora_finetune import build_lora_model, fine_tune, score

model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
examples = build_training_examples()
held_out_examples = build_examples_for_report(SAMPLE_SET_DIR / HELD_OUT_REPORT)

Before fine-tuning: confirm the base model's training and held-out baselines match Chapter 3's numbers.

In [ ]:
print("Training baseline:", score(run_baseline(model, tokenizer, examples)))
print("Held-out baseline:", score(run_baseline(model, tokenizer, held_out_examples)))

Fine-tune, then compare training recall against held-out generalization.

In [ ]:
lora_model = build_lora_model(model)
lora_model.print_trainable_parameters()
fine_tune(lora_model, tokenizer, examples)

print("Training recall:", score(run_baseline(lora_model, tokenizer, examples)))
print("Held-out generalization:", score(run_baseline(lora_model, tokenizer, held_out_examples)))

Look at exactly which answers are wrong, and where each borrowed answer actually comes from.

In [ ]:
for r in run_baseline(lora_model, tokenizer, held_out_examples):
    print(f"Q: {r['instruction']} ({r['input']})")
    print(f"Expected: {r['expected']}")
    print(f"Model said: {r['base_model_answer']}\n")